# Colombia Open — Departamentos

Este notebook muestra cómo cargar el dataset **`departamentos`** (DIVIPOLA, DANE) con `colombia-open` y explora su esquema: código DIVIPOLA de 2 dígitos, nombre oficial, y la columna normalizada para joins/búsqueda.

> **Nota (2026-10):** el proyecto todavía no está publicado en PyPI ni tiene un Release con datos pre-construidos (ver `README.md` del repo, sección "Estado del proyecto"). Por eso este notebook clona el repo e instala el extra `pipeline` para generar los datos localmente en vez de hacer `pip install colombia-open`. Cuando exista un Release publicado, la celda de setup se reduce a `pip install colombia-open`.

## Setup

In [ ]:
!git clone --depth 1 https://github.com/davidcasr/colombia-open.git
%cd colombia-open
!pip install -q -e ".[pipeline]"
!python -m src.build_pipeline

## Cargar el dataset

In [ ]:
from colombia_open import ColombiaOpen

hub = ColombiaOpen()
departamentos = hub.load_polars("departamentos")

print(departamentos.shape)
departamentos.head(10)

## Esquema

| Columna | Tipo | Descripción |
|---|---|---|
| `cod_dpto` | string (2 dígitos) | Código DIVIPOLA de departamento. **Nunca se convierte a int** — pierde el cero inicial (ej. Antioquia es `"05"`, no `5`). |
| `nombre_dpto` | string | Nombre oficial, como lo publica el DANE. |
| `nombre_dpto_clean` | string | `nombre_dpto` normalizado (minúsculas, sin tildes, sin puntuación) — para joins/búsqueda determinística. |

In [ ]:
# El código siempre mantiene el ancho fijo de 2 dígitos
departamentos.select(["cod_dpto", "nombre_dpto"]).sort("cod_dpto").head(5)

## Buscar un departamento por nombre normalizado

In [ ]:
from colombia_open.text import clean_name

objetivo = clean_name("Bogotá, D.C.")
departamentos.filter(departamentos["nombre_dpto_clean"] == objetivo)

## Siguiente paso

Ver [`02_municipios.ipynb`](02_municipios.ipynb) para el join `departamentos` ↔ `municipios` y la resolución de nombres de ciudades.

Documentación completa del esquema: [`docs/datasets/departamentos.md`](../../docs/datasets/departamentos.md).